<a href="https://colab.research.google.com/github/RAJAMURUGAN-VS/genai-learning-journey/blob/main/02-groq-api/04_complete_function_calling_workflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install Groq package

In [19]:
!pip install groq

Import necessary classes and functions

In [20]:
from google.colab import userdata
from groq import Groq
from pprint import pprint
import json
import requests

Create client object using Groq class

In [21]:
client=Groq(
  api_key=userdata.get('GROQ_API_KEY')
)
print(client)

Function to get Weather Information

In [22]:
def get_weather(location):
 api_key = userdata.get('WEATHER_API_KEY')
 url = f"http://api.openweathermap.org/data/2.5/weather?q={location}&units=metric&appid={api_key}"
 response = requests.get(url)
 data = response.json()
 pprint(data)

 if data["cod"] == 200:
   return {
     "location": location,
     "temperature": data["main"]["temp"],
     "description": data["weather"][0]["description"]
   }
 else:
   return {"Oops! Something went wrong."}

Tool Definition for get_weather function

In [23]:
tools = [
  {
    "type": "function",
    "function": {
      "name": "get_weather",
      "description": "Get current weather for a city",
      "parameters": {
        "type": "object",
        "properties": {
          "location": {
            "type": "string",
            "description": "City name like Mumbai, London"
            }
            },
      "required": ["location"]
           }
       }
   }
]

Sending the Tool Definition to the Model and Ask Weather Updates from Llama

In [24]:
llm_messages = [
  {
    "role": "system",
    "content": "You are a weather assistant. Use get_weather function when asked about weather."
  },
  {
    "role": "user",
    "content": "What's the weather in Mumbai?"
  }
]

response = client.chat.completions.create(
  model="llama-3.3-70b-versatile",
  messages=llm_messages,
  tools=tools,
  tool_choice="auto"
)

print(response.model_dump_json(indent=2))

{
  "id": "chatcmpl-1eaf4e1e-8c7c-4860-9fd0-91254d17ae7c",
  "choices": [
    {
      "finish_reason": "tool_calls",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": null,
        "role": "assistant",
        "annotations": null,
        "executed_tools": null,
        "function_call": null,
        "reasoning": null,
        "tool_calls": [
          {
            "id": "20x5zbh34",
            "function": {
              "arguments": "{\"location\":\"Mumbai\"}",
              "name": "get_weather"
            },
            "type": "function"
          }
        ]
      }
    }
  ],
  "created": 1779775075,
  "model": "llama-3.3-70b-versatile",
  "object": "chat.completion",
  "mcp_list_tools": null,
  "service_tier": "on_demand",
  "system_fingerprint": "fp_0761e44d7b",
  "usage": {
    "completion_tokens": 15,
    "prompt_tokens": 243,
    "total_tokens": 258,
    "completion_time": 0.043324621,
    "completion_tokens_details": null,
    "prompt_time"

In [25]:
response_message = response.choices[0].message
print(response_message.model_dump_json(indent=2))

{
  "content": null,
  "role": "assistant",
  "annotations": null,
  "executed_tools": null,
  "function_call": null,
  "reasoning": null,
  "tool_calls": [
    {
      "id": "20x5zbh34",
      "function": {
        "arguments": "{\"location\":\"Mumbai\"}",
        "name": "get_weather"
      },
      "type": "function"
    }
  ]
}


First API Call and Handling the Response

In [26]:
if response_message.tool_calls:
  tool_call = response_message.tool_calls[0]
  arguments = json.loads(tool_call.function.arguments)
  location = arguments['location']
  weather_data = get_weather(location)
  print(f"Weather data at {location}: {weather_data}")

{'base': 'stations',
 'clouds': {'all': 40},
 'cod': 200,
 'coord': {'lat': 19.0144, 'lon': 72.8479},
 'dt': 1779774987,
 'id': 1275339,
 'main': {'feels_like': 40.99,
          'grnd_level': 1010,
          'humidity': 59,
          'pressure': 1011,
          'sea_level': 1011,
          'temp': 33.99,
          'temp_max': 33.99,
          'temp_min': 33.94},
 'name': 'Mumbai',
 'sys': {'country': 'IN',
         'id': 9052,
         'sunrise': 1779755487,
         'sunset': 1779802807,
         'type': 1},
 'timezone': 19800,
 'visibility': 6000,
 'weather': [{'description': 'haze', 'icon': '50d', 'id': 721, 'main': 'Haze'}],
 'wind': {'deg': 280, 'speed': 5.66}}
Weather data at Mumbai: {'location': 'Mumbai', 'temperature': 33.99, 'description': 'haze'}


Send the Results Back to the LLM

In [30]:
  llm_messages.append(response_message)

  llm_messages.append({
    "role": "tool",
    "tool_call_id": tool_call.id,
    "content": json.dumps(weather_data)
  })

  pprint(llm_messages)

[{'content': 'You are a weather assistant. Use get_weather function when asked '
             'about weather.',
  'role': 'system'},
 {'content': "What's the weather in Mumbai?", 'role': 'user'},
 ChatCompletionMessage(content=None, role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning=None, tool_calls=[ChatCompletionMessageToolCall(id='20x5zbh34', function=Function(arguments='{"location":"Mumbai"}', name='get_weather'), type='function')]),
 {'content': '{"location": "Mumbai", "temperature": 33.99, "description": '
             '"haze"}',
  'role': 'tool',
  'tool_call_id': '20x5zbh34'},
 ChatCompletionMessage(content=None, role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning=None, tool_calls=[ChatCompletionMessageToolCall(id='20x5zbh34', function=Function(arguments='{"location":"Mumbai"}', name='get_weather'), type='function')]),
 {'content': '{"location": "Mumbai", "temperature": 33.99, "description": '
             '

Sending Tool Output and Getting the Final Response

In [28]:
  final_response = client.chat.completions.create(
    messages = llm_messages,
    model = "llama-3.3-70b-versatile",
    tools = tools,
    tool_choice = "auto"
  )

  print(final_response.choices[0].message.content)

The current weather in Mumbai is haze with a temperature of 33.99 degrees.
